In [31]:
import numpy as np
import pandas as pd
import os
import joblib
import pickle
import math
import ast
from scipy.stats import median_abs_deviation, hypergeom, mannwhitneyu
from scipy.cluster.hierarchy import linkage, dendrogram, leaves_list
from scipy.spatial.distance import squareform
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
# Saving plots with editable text
plt.rcParams['pdf.fonttype'] = 42  # TrueType fonts (editable text)

In [32]:
import sys
# Ensure this analysis directory is importable regardless of kernel CWD
_here = '/projects/bhdw/asachan/methods/FIREFate/multiome_dynamic_regulation/py_scripts/analysis'
if _here not in sys.path:
    sys.path.insert(0, _here)

import dictys
from utils_custom import *
from pseudotime_curves import *
from episodic_dynamics import *
from config import *

In [33]:
import importlib
import firefate.utils.plots, firefate.utils.custom
import temporal_clustering
# reload firefate helpers first, then temporal_clustering so it re-binds fresh names
importlib.reload(firefate.utils.plots)
importlib.reload(firefate.utils.custom)
importlib.reload(temporal_clustering)
from temporal_clustering import StateFrequency, TFForceWaves, TFForceValidation

In [4]:
config = Config()

In [5]:
# Load data
dictys_dynamic_object = dictys.net.dynamic_network.from_file('/work/nvme/bhdw/asachan/data_files/firefate/bcell/outs/dynamic.h5')

### Defining lineage trajectories

In [6]:
PB_fate_window_indices = [1] + list(range(97, 3, -1)) + [0] + list(range(98, 147, 1)) + [2]
GC_fate_window_indices = [1] + list(range(97, 3, -1)) + [0] + list(range(147, 193, 1)) + [3]
PB_post_bifurcation_window_indices = [0] + list(range(98, 147, 1)) + [2]
GC_post_bifurcation_window_indices = [0] + list(range(147, 193, 1)) + [3]

In [7]:
# Define distinct colors for better visibility
colors_cell_count = {
    'ActB-1': '#87CEFA',     # lightskyblue
    'ActB-2': '#1E90FF',     # dodgerblue
    'ActB-3': '#00008B',     # darkblue
    'ActB-4': '#9370DB',     # mediumorchid
    'GC-1': '#7BDE7B',       # custom light green
    'GC-2': '#008000',       # green
    'PB-2': '#BB3636',       # custom red
    'earlyActB': '#008080',   # teal
    'earlyPB': '#F08080'   # lightcoral
}

## TF forces

In [34]:
# TF forces over pseudotime (expression / regulation curves cached internally)
waves_pb = TFForceWaves(
    dictys_dynamic_object,
    trajectory_range=(1, 2),
    num_points=100,
    dist=0.0005,
    sparsity=0.01,
)

In [35]:
# GC branch: same TF-forces machinery over the GC trajectory (node 0 -> 3).
waves_gc = TFForceWaves(
    dictys_dynamic_object,
    trajectory_range=(1, 3),
    num_points=100,
    dist=0.0005,
    sparsity=0.01,
)

### FIREFate state-specific enriched links and their forces across both lineages

In [36]:
ss_firefate_combined = pd.read_csv('/projects/bhdw/asachan/tmp/ss_firefate_links_2B.csv')

In [37]:
display(ss_firefate_combined)

,source,target,key,weight,cluster,strength,case
0,PRDM1,RUNX2,1,0.293075,3,1,slide
1,PRDM1,IQGAP2,1,0.219831,3,1,slide
2,BACH2,XBP1,0,-0.229950,7,1,slide
3,BACH2,MZB1,1,-0.234813,3,1,slide
4,BACH2,JCHAIN,1,-0.258168,3,1,slide
...,...,...,...,...,...,...,...
73,PBX3,CDK6,1,0.231756,3,1,slide
74,PBX3,AFF3,1,0.332018,3,1,slide
75,ARID5B,CPEB4,1,0.237878,3,1,slide
76,ARID5B,PDE4D,0,0.228636,7,1,slide


In [38]:
# Build (TF, target) tuples, keeping only links whose TF and target are both
# present in the dictys object. get_beta_curves can only compute forces for genes
# in the GRN (TFs in nids[0], targets in ndict); filtering here makes the computed
# link set explicit rather than relying on the internal skip.
all_links = list(zip(ss_firefate_combined['source'], ss_firefate_combined['target']))

_, _, missing_tfs = get_tf_indices(dictys_dynamic_object, list({tf for tf, _ in all_links}))
missing_tfs = set(missing_tfs)
ndict = dictys_dynamic_object.ndict

ss_firefate_combined_tuple = [(tf, tg) for tf, tg in all_links
                     if tf not in missing_tfs and tg in ndict]
dropped = [(tf, tg) for tf, tg in all_links
           if tf in missing_tfs or tg not in ndict]
print(f"{len(ss_firefate_combined_tuple)}/{len(all_links)} links kept; "
      f"{len(dropped)} dropped (TF/target absent from GRN): {dropped}")

66/78 links kept; 12 dropped (TF/target absent from GRN): [('AFF1', 'MAN1A1'), ('AFF1', 'FNDC3A'), ('AFF1', 'CEP128'), ('HERPUD1', 'TRAM1'), ('HERPUD1', 'IRF4'), ('HERPUD1', 'BTG2'), ('HERPUD1', 'PAX5'), ('HIVEP2', 'IRF4'), ('HIVEP2', 'BTG2'), ('HIVEP2', 'GAB1'), ('HIVEP2', 'GLCCI1'), ('HIVEP2', 'CPEB4')]


## Cross-branch pooled comparison (PB ∪ GC)

In [ ]:
# The 66 enriched links act on both the PB and GC branches, so collapsing a single
# branch across phases under-separates them. Score each enriched link on BOTH branches
# and keep its abs-max TF force across branches; the random null is the UNION of the
# per-branch random pools (each random link keeps its own single-branch force, not a
# cross-branch max), size-matched to the enriched links. The result keeps a
# 'branch' column = the lineage where each enriched link's force was strongest.

xval = TFForceValidation(
    {'PB': waves_pb, 'GC': waves_gc},
    enriched_links=ss_firefate_combined_tuple,
    varname='w_in',
    mode='combined',
)
xval_df = xval.run(exclude='tf_and_target', random_state=0)
display(xval_df)

fig, ax = xval.plot(ylabel='abs(max TF-force)')
# save as svg with editable text
#fig.savefig('/projects/bhdw/asachan/papers/firefate/figures/state_specific_links_validation.svg', format='svg', bbox_inches='tight')
plt.show()

# Validation of episodic links

In [25]:
# load the episodically enriched links from file
episodic_links_file = '/projects/bhdw/asachan/papers/firefate/figures/enriched_tf_lf_targets_per_episode.csv'
episodic_links = pd.read_csv(episodic_links_file)

In [27]:
display(episodic_links)

,lineage,episode,TF,p_value,genes_in_lf,n_genes_in_lf
0,GC,Ep4,CDC5L,0.003546,"ARNTL2, MZB1, TNFAIP8, UBAC2",4
1,GC,Ep1,CREB3L2,0.009150,"FNDC3A, HSP90B1, LMAN1, SSR1, TRAM1, TXNDC5",6
2,GC,Ep1,IKZF3,0.002195,"ANKRD28, CD74, PIKFYVE, RNF213",4
3,GC,Ep3,IRF7,0.005185,"B2M, CEP128",2
4,GC,Ep4,IRF7,0.001808,"B2M, CEP128",2
5,GC,Ep4,MEF2C,0.000220,"IRF4, SEL1L3, SLA, TNFAIP8",4
6,GC,Ep4,POU2F1,0.006940,"CCSER2, DEK, EEA1, IRF4, MAPK1, SLC25A13",6
7,GC,Ep3,USF2,0.001520,"FNDC3A, HSP90B1, LMAN1, RNF213, SSR1, SUB1, TRAM1",7
8,GC,Ep4,USF2,0.004210,"FNDC3A, HSP90B1, LMAN1, RNF213, SSR1, SUB1, TRAM1",7
9,PB,Ep1,CREB3L2,0.000220,"FNDC3A, HSP90B1, LMAN1, SSR1, TRAM1",5


In [28]:
#create tuples from the TF and genes_in_lf list of comma separated targets and take union of all links across all rows to make a unique set of links
episodic_links_tuples = set()
for _, row in episodic_links.iterrows():
    tf = row['TF']
    targets = row['genes_in_lf'].split(',')
    for target in targets:
        episodic_links_tuples.add((tf, target.strip()))


In [30]:
display(len(episodic_links_tuples))

55